# Web Server Log Analysis - Python Take-Home Assessment

## Overview
This assessment involves analyzing the Calgary HTTP dataset, which contains approximately one year's worth of HTTP requests to the University of Calgary's Computer Science web server. You'll work with real-world web server log data to extract meaningful insights and demonstrate your Python data analysis skills.

## Part 1: Data Loading and Cleaning

### Instructions

* Work in the cells below - You can add as many cells as needed for data loading, cleaning, and exploration
* Import required libraries
* Implement data loading and cleaning - Create functions to download, parse, and clean the log data
* Explore the data - Understand the structure and identify any data quality issues

###Step 1 Import Required Libararies

In [1]:
import gzip
import re
import requests
import pandas as pd
from datetime import datetime
from collections import Counter


### Step 2 Function to Download Dataset

In [2]:
def download_dataset(url, output_path):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()

        with open(output_path, 'wb') as file:
            for chunk in response.iter_content(chunk_size=1024):
                if chunk:
                    file.write(chunk)

        print("Download completed successfully.")

    except Exception as e:
        print("Error while downloading file:", e)

In [3]:
url = "http://ita.ee.lbl.gov/traces/calgary_access_log.gz"
output_path = "calgary_access_log.gz"

download_dataset(url, output_path)

Download completed successfully.


### Step 3 Function to load and read .gz File

In [4]:
def load_logs(file_path):
    logs = []

    try:
        with gzip.open(file_path, 'rt', encoding='latin-1') as file:
            for line in file:
                logs.append(line.strip())

        print("File loaded successfully. Total logs:", len(logs))

    except Exception as e:
        print("Error while loading file:", e)

    return logs

logs = load_logs("calgary_access_log.gz")

File loaded successfully. Total logs: 726739


### Step 4 Function to parse logs(Regex)

In [5]:
def parse_logs(logs):
    pattern = re.compile(r'(\S+) - - \[(.*?)\] "(\S+) (.*?) (\S+)" (\d{3}) (\S+)')

    parsed_data = []

    for log in logs:
        try:
            match = pattern.match(log)

            if match:
                host, timestamp, method, resource, protocol, status, bytes_ = match.groups()

                parsed_data.append({
                    "host": host,
                    "timestamp": timestamp,
                    "method": method,
                    "resource": resource,
                    "status": int(status),
                    "bytes": 0 if bytes_ == '-' else int(bytes_)
                })

        except Exception as e:
            continue  # skip bad lines

    print("Parsed records:", len(parsed_data))
    return pd.DataFrame(parsed_data)
df = parse_logs(logs)
df.head()


Parsed records: 723236


,host,timestamp,method,resource,status,bytes
0,local,24/Oct/1994:13:41:41 -0600,GET,index.html,200,150
1,local,24/Oct/1994:13:41:41 -0600,GET,1.gif,200,1210
2,local,24/Oct/1994:13:43:13 -0600,GET,index.html,200,3185
3,local,24/Oct/1994:13:43:14 -0600,GET,2.gif,200,2555
4,local,24/Oct/1994:13:43:15 -0600,GET,3.gif,200,36403


### Step 6 Data Cleaning Function

In [6]:
def clean_data(df):
    initial_count = len(df)

    # Convert timestamp
    df['datetime'] = pd.to_datetime(
        df['timestamp'],
        format="%d/%b/%Y:%H:%M:%S %z",
        errors='coerce',
        utc=True
    )

    # Convert numeric fields
    df['status'] = pd.to_numeric(df['status'], errors='coerce')
    df['bytes'] = pd.to_numeric(df['bytes'], errors='coerce')

    # Remove invalid rows (STRICT)
    df = df[
        df['datetime'].notna() &                      # valid timestamp
        df['status'].between(100, 599) &              # valid HTTP status
        df['resource'].notna() &                      # resource exists
        (df['resource'].str.len() > 0)                # non-empty resource
    ]

    # Handle bytes
    df['bytes'] = df['bytes'].fillna(0)

    # Remove suspicious resources (IMPORTANT improvement)
    df = df[~df['resource'].str.contains(r'\s', na=False)]  # remove spaces

    # Extract fields
    df['date'] = df['datetime'].dt.strftime('%d-%b-%Y')
    df['hour'] = df['datetime'].dt.hour

    df['resource'] = df['resource'].apply(lambda x: x.split('?')[0])

    df['extension'] = df['resource'].apply(
        lambda x: x.split('.')[-1] if '.' in x else 'unknown'
    )

    final_count = len(df)

    print(f"Rows before cleaning: {initial_count}")
    print(f"Rows after cleaning: {final_count}")
    print(f"Removed rows: {initial_count - final_count}")

    return df
df=clean_data(df)

len(df)


Rows before cleaning: 723236
Rows after cleaning: 722241
Removed rows: 995


722241

## ⚠️ IMPORTANT: Template Questions Section
**DO NOT MODIFY THE TEMPLATE BELOW THIS POINT**

The following section contains the assessment questions. You may add cells above this section for data loading, cleaning, and exploration, but do not modify the function signatures or structure of the questions below.

## Part 2: Analysis Questions

### Instructions

* Implement each function according to its docstring specifications
* Use the cleaned data you prepared in Part 1
* Ensure your functions return the exact data types specified
* Test your functions to verify they work correctly
* You may add helper functions, but keep the main function signatures unchanged

### Q1: Count of total log records

In [7]:
def total_log_records() -> int:
    """
    Q1: Count of total log records.

    Objective:
        Determine the total number of HTTP log entries in the dataset.
        Each line in the log file represents one HTTP request.

    Returns:
        int: Total number of log entries.
    """

    # TODO: Implement logic to count log records

    return len(df)
      # Placeholder return


answer1 = total_log_records()
print("Answer 1:")
print(answer1)

Answer 1:
722241


### Q2: Count of unique hosts

In [8]:
def unique_host_count() -> int:
    """
    Q2: Count of unique hosts.

    Objective:
        Determine how many distinct hosts accessed the server.

    Returns:
        int: Number of unique hosts.
    """

    # TODO: Implement logic to count unique hosts

    return df['host'].nunique()  # Placeholder return


answer2 = unique_host_count()
print("Answer 2:")
print(answer2)

Answer 2:
2


### Q3: Date-wise unique filename counts

In [9]:
def datewise_unique_filename_counts() -> dict[str, int]:
    """
    Q3: Date-wise unique filename counts.

    Objective:
        For each date, count the number of unique filenames that accessed the server.
        The date should be in 'dd-MMM-yyyy' format (e.g., '01-Jul-1995').

    Returns:
        dict: A dictionary mapping each date to its count of unique filenames.
              Example: {'01-Jul-1995': 123, '02-Jul-1995': 150}
    """

    # TODO: Implement logic for date-wise unique filename counts

    return df.groupby('date')['resource'].nunique().to_dict() # Placeholder return


answer3 = datewise_unique_filename_counts()
print("Answer 3:")
print(answer3)

Answer 3:
{'01-Apr-1995': 407, '01-Aug-1995': 663, '01-Dec-1994': 244, '01-Feb-1995': 571, '01-Jan-1995': 82, '01-Jul-1995': 405, '01-Jun-1995': 569, '01-Mar-1995': 519, '01-May-1995': 432, '01-Nov-1994': 436, '01-Oct-1995': 583, '01-Sep-1995': 445, '02-Apr-1995': 394, '02-Aug-1995': 785, '02-Dec-1994': 356, '02-Feb-1995': 619, '02-Jan-1995': 128, '02-Jul-1995': 362, '02-Jun-1995': 546, '02-Mar-1995': 676, '02-May-1995': 694, '02-Nov-1994': 422, '02-Oct-1995': 839, '02-Sep-1995': 306, '03-Apr-1995': 813, '03-Aug-1995': 659, '03-Dec-1994': 152, '03-Feb-1995': 566, '03-Jan-1995': 257, '03-Jul-1995': 460, '03-Jun-1995': 397, '03-Mar-1995': 502, '03-May-1995': 565, '03-Nov-1994': 450, '03-Oct-1995': 833, '03-Sep-1995': 207, '04-Apr-1995': 862, '04-Aug-1995': 739, '04-Dec-1994': 211, '04-Feb-1995': 450, '04-Jan-1995': 313, '04-Jul-1995': 491, '04-Jun-1995': 323, '04-Mar-1995': 364, '04-May-1995': 700, '04-Nov-1994': 417, '04-Oct-1995': 909, '04-Sep-1995': 309, '05-Apr-1995': 830, '05-Aug-19

### Q4: Number of 404 response codes

In [10]:
def count_404_errors() -> int:
    """
    Q4: Number of 404 response codes.

    Objective:
        Count how many times the HTTP 404 Not Found status appears in the logs.

    Returns:
        int: Number of 404 errors.
    """

    # TODO: Implement logic to count 404 errors

    return (df['status']==404).sum()  # Placeholder return


answer4 = count_404_errors()
print("Answer 4:")
print(answer4)

Answer 4:
23440


### Q5: Top 15 filenames with 404 responses

In [11]:
def top_15_filenames_with_404() -> list[tuple[str, int]]:
    """
    Q5: Top 15 filenames with 404 responses.

    Objective:
        Identify which requested URLs most frequently resulted in a 404 error.
        Return the top 15 filenames sorted by frequency.

    Returns:
        list: A list of tuples (filename, count), sorted by count in descending order.
              Example: [('index.html', 200), ...]
    """

    # TODO: Implement logic to find top 15 filenames with 404

    return list(df[df['status']==404]['resource'].value_counts().head(15).items()) # Placeholder return


answer5 = top_15_filenames_with_404()
print("Answer 5:")
print(answer5)

Answer 5:
[('index.html', 4692), ('4115.html', 900), ('1611.html', 649), ('5698.xbm', 585), ('710.txt', 408), ('2002.html', 258), ('2177.gif', 193), ('10695.ps', 161), ('6555.html', 153), ('487.gif', 152), ('151.html', 149), ('488.gif', 148), ('40.html', 148), ('3414.gif', 148), ('9678.gif', 142)]


### Q6: Top 15 file extension with 404 responses

In [12]:
def top_15_ext_with_404() -> list[tuple[str, int]]:
    """
    Q6: Top 15 file extensions with 404 responses.

    Objective:
        Find which file extensions generated the most 404 errors.
        Return the top 15 sorted by number of 404s.

    Returns:
        list: A list of tuples (extension, count), sorted by count in descending order.
              Example: [('html', 45), ...]
    """

    # TODO: Implement logic to find top 15 extensions with 404

    return list(df[df['status']==404]['extension'].value_counts().head(15).items())  # Placeholder return


answer6 = top_15_ext_with_404()
print("Answer 6:")
print(answer6)

Answer 6:
[('html', 12139), ('gif', 7202), ('xbm', 824), ('ps', 754), ('jpg', 520), ('txt', 496), ('GIF', 135), ('htm', 107), ('cgi', 77), ('gif"', 45), ('com', 45), ('Z', 41), ('dvi', 40), ('com/', 37), ('ca', 36)]


### Q7: Total bandwidth transferred per day for the month of July 1995

In [13]:
def total_bandwidth_per_day() -> dict[str, int]:
    """
    Q7: Total bandwidth transferred per day for the month of July 1995.

    Objective:
        Sum the number of bytes transferred per day.
        Skip entries where the byte field is missing or '-'.

    Returns:
        dict: A dictionary mapping each date to total bytes transferred.
              Example: {'01-Jul-1995': 123456789, ...}
    """

    # TODO: Implement logic to compute total bandwidth per day
    df_july=df[df['datetime'].dt.strftime('%b-%Y')=='Jul-1995']
    return df_july.groupby('date')['bytes'].sum().to_dict()  # Placeholder return


answer7 = total_bandwidth_per_day()
print("Answer 7:")
print(answer7)

Answer 7:
{'01-Jul-1995': 16986893, '02-Jul-1995': 7892436, '03-Jul-1995': 11739190, '04-Jul-1995': 24976177, '05-Jul-1995': 22468066, '06-Jul-1995': 20419373, '07-Jul-1995': 9566244, '08-Jul-1995': 5475250, '09-Jul-1995': 4312672, '10-Jul-1995': 13195178, '11-Jul-1995': 22567659, '12-Jul-1995': 17802591, '13-Jul-1995': 15959344, '14-Jul-1995': 16143956, '15-Jul-1995': 17900110, '16-Jul-1995': 8086988, '17-Jul-1995': 18423405, '18-Jul-1995': 17947142, '19-Jul-1995': 16164044, '20-Jul-1995': 25444995, '21-Jul-1995': 25910651, '22-Jul-1995': 6224677, '23-Jul-1995': 10089651, '24-Jul-1995': 20554069, '25-Jul-1995': 23269918, '26-Jul-1995': 26191814, '27-Jul-1995': 22953465, '28-Jul-1995': 37450120, '29-Jul-1995': 16285535, '30-Jul-1995': 21147546, '31-Jul-1995': 29820800}


### Q8: Hourly request distribution

In [14]:
def hourly_request_distribution() -> dict[int, int]:
    """
    Q8: Hourly request distribution.

    Objective:
        Count the number of requests made during each hour (00 to 23).
        Useful for understanding traffic peaks.

    Returns:
        dict: A dictionary mapping hour (int) to request count.
              Example: {0: 120, 1: 90, ..., 23: 80}
    """

    # TODO: Implement logic for hourly distribution

    return df['hour'].value_counts().sort_index().to_dict()  # Placeholder return


answer8 = hourly_request_distribution()
print("Answer 8:")
print(answer8)

Answer 8:
{0: 39499, 1: 32525, 2: 30636, 3: 28075, 4: 25914, 5: 22744, 6: 19705, 7: 16916, 8: 13808, 9: 11381, 10: 10536, 11: 10366, 12: 12261, 13: 15145, 14: 22029, 15: 30858, 16: 37914, 17: 46204, 18: 45650, 19: 49935, 20: 50956, 21: 52780, 22: 50318, 23: 46086}


### Q9: Top 10 most requested filenames

In [15]:
def top_10_most_requested_filenames() -> list[tuple[str, int]]:
    """
    Q9: Top 10 most requested filenames.

    Objective:
        Identify the most commonly requested URLs (irrespective of status code).

    Returns:
        list: A list of tuples (filename, count), sorted by count in descending order.
                Example: [('index.html', 500), ...]
    """

    # TODO: Implement logic to find top 10 most requested filenames

    return list(df['resource'].value_counts().head(10).items())  # Placeholder return


answer9 = top_10_most_requested_filenames()
print("Answer 9:")
print(answer9)

Answer 9:
[('index.html', 139505), ('3.gif', 24006), ('2.gif', 23595), ('4.gif', 8018), ('244.gif', 5148), ('5.html', 5009), ('4097.gif', 4874), ('8870.jpg', 4492), ('6733.gif', 4278), ('8472.gif', 3843)]


### Q10: HTTP response code distribution

In [16]:
def response_code_distribution() -> dict[int, int]:
    """
    Q10: HTTP response code distribution.

    Objective:
        Count how often each HTTP status code appears in the logs.

    Returns:
        dict: A dictionary mapping HTTP status codes (as int) to their frequency.
              Example: {200: 150000, 404: 3000}
    """

    # TODO: Implement logic for response code counts

    return df['status'].value_counts().to_dict()  # Placeholder return


answer10 = response_code_distribution()
print("Answer 10:")
print(answer10)

Answer 10:
{200: 566122, 304: 97560, 302: 30258, 404: 23440, 403: 4740, 501: 43, 401: 43, 500: 22, 400: 13}
